# 23 (ML) — LogicalVector & Planner Metadata

**Framework perspective.** Demonstrates the ml_scope §8/§9/§41 wiring: estimators accept `featuresCol` as a `LogicalVector`, the `MLSemanticPlanner` assigns a backend to each stage, and pipelines expose stage backends and logical vectors. Pure metadata — SQL behavior is unchanged.

In [ ]:
import os
from dotenv import load_dotenv
from irispark import IrisParkSession

load_dotenv()

# Connection via environment variables (matches examples/basic_usage.py).
# Set IRIS_HOST / IRIS_PORT / IRIS_NAMESPACE / IRIS_USERNAME / IRIS_PASSWORD.
try:
    session = IrisParkSession.builder() \
        .host(os.environ.get("IRIS_HOST", "localhost")) \
        .port(int(os.environ.get("IRIS_PORT", 1972))) \
        .namespace(os.environ.get("IRIS_NAMESPACE", "USER")) \
        .username(os.environ.get("IRIS_USERNAME", "_SYSTEM")) \
        .password(os.environ.get("IRIS_PASSWORD", "SYS")) \
        .getOrCreate()
    print("Connected to IRIS:", session)
except Exception as e:
    print("SKIP: IRIS not reachable -", e)
    session = None

In [ ]:
if session is None:
    raise SystemExit("IRIS not reachable; skipping this notebook.")

## 1. Synthetic regression data

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
X = rng.normal(size=(200, 2))
y = 2.0 + 3.0 * X[:, 0] - 1.0 * X[:, 1] + rng.normal(scale=0.1, size=200)
df = session.createDataFrame(pd.DataFrame({"x1": X[:, 0], "x2": X[:, 1], "label": y}))
df.show(3)

## 2. Classic form — `featuresCol` as a list

In [ ]:
from irispark.ml.regression import LinearRegression

lr = LinearRegression(featuresCol=["x1", "x2"], labelCol="label")
model = lr.fit(df)
pred = model.transform(df)
pred.select("x1", "x2", "label", "prediction").show(3)
print("backend:", model.backend)

## 3. LogicalVector form — same result, plus metadata

`featuresCol` also accepts a `LogicalVector`; the estimator resolves it to the column list and retains the descriptor on the fitted model.

In [ ]:
from irispark.ml.linalg import LogicalVector

lv = LogicalVector(["x1", "x2"])
lr2 = LinearRegression(featuresCol=lv, labelCol="label")
model2 = lr2.fit(df)
print("resolved featuresCol:", lr2.getFeaturesCol())
print("estimator.logicalVector:", lr2.logicalVector)
print("model.logicalVector:", model2.logicalVector)
print("model.backend:", model2.backend)
model2.transform(df).select("label", "prediction").show(3)

## 4. Planner backend resolution

The `MLSemanticPlanner` assigns a backend per stage: feature transformers → SQL, numpy estimators → Python, AutoML → IntegratedML.

In [ ]:
from irispark.ml.planner import default_planner
from irispark.ml.feature import VectorAssembler, StandardScaler
from irispark.ml.classification import LogisticRegression
from irispark.ml.automl import AutoMLClassifier

print("VectorAssembler ->", default_planner.resolve_backend(VectorAssembler(inputCols=["x1"], outputCol="f")))
print("StandardScaler ->", default_planner.resolve_backend(StandardScaler(inputCol="x1", outputCol="s")))
print("LinearRegression ->", default_planner.resolve_backend(LinearRegression(featuresCol=["x1"], labelCol="y")))
print("LogisticRegression ->", default_planner.resolve_backend(LogisticRegression(featuresCol=["x1"], labelCol="y")))
print("AutoMLClassifier ->", default_planner.resolve_backend(AutoMLClassifier(featuresCol=["x1"], labelCol="y")))

## 5. Pipeline integration

`Pipeline.fit` records each stage's backend; `PipelineModel.getBackends()` and `getLogicalVectors()` expose the metadata.

In [ ]:
from irispark.ml.pipeline import Pipeline

scaled = StandardScaler(inputCol="x1", outputCol="x1_s")
va = VectorAssembler(inputCols=["x1_s", "x2"], outputCol="features")
lr3 = LogisticRegression(featuresCol=["x1_s", "x2"], labelCol="label")
pipe = Pipeline(stages=[scaled, va, lr3])
pipe_model = pipe.fit(df)
print("stage backends:", pipe_model.getBackends())
print("logical vectors:", pipe_model.getLogicalVectors())
pipe_model.transform(df).select("label", "prediction").show(3)

In [ ]:
if session is not None:
    session.close()
    print("Session closed.")